The goal is to train and visualize the outputs of a simple Deep Convolutional GAN (DCGAN) to generate realistic-looking images of clothing.

In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras import layers


In [ ]:
tf.random.set_seed(42)
np.random.seed(42)

3a. Use the FashionMNIST training dataset (which we used in previous assignments) to train the DCGAN.
Images are grayscale and size 28 ×28.

In [ ]:
# Load the data
(x_train,y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

In [ ]:
# Reshape to include channel dimension: [batch_size, height, width, channels]
x_train = x_train.reshape(x_train.shape[0], 28, 28, 1)
x_test = x_test.reshape(x_test.shape[0], 28, 28, 1)

In [ ]:
# Set hyperparameters
BATCH_SIZE = 64  # As suggested in the assignment
EPOCHS = 50
z_dim = 100  # Dimensionality of the noise vector
images_generated = 9  # Number of images to display

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(60000).batch(BATCH_SIZE)

In [ ]:
# Define the generator model
def build_generator():
    model = tf.keras.Sequential()

    # Dense layer that maps the noise vector to 7×7×256
    model.add(layers.Dense(7*7*256, use_bias=False, input_shape=(z_dim,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.3))

    # Reshape to 7×7×256 for convolutional layers
    model.add(layers.Reshape((7, 7, 256)))

    # First transpose conv: 256×7×7 → 128×7×7 (maintain size)
    model.add(layers.Conv2DTranspose(128, kernel_size=5, strides=1, padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.3))

    # Second transpose conv: 128×7×7 → 64×14×14 (double size)
    model.add(layers.Conv2DTranspose(64, kernel_size=5, strides=2, padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(alpha=0.3))

    # Final transpose conv: 64×14×14 → 1×28×28 (double size)
    model.add(layers.Conv2DTranspose(1, kernel_size=5, strides=2, padding='same', use_bias=False, activation='tanh'))

    return model

In [ ]:
def build_discriminator():
    model = tf.keras.Sequential()

    # First conv: 1×28×28 → 64×14×14
    model.add(layers.Conv2D(64, kernel_size=5, strides=2, padding='same', input_shape=[28, 28, 1]))
    model.add(layers.LeakyReLU(alpha=0.3))
    model.add(layers.Dropout(0.3))

    # Second conv: 64×14×14 → 128×7×7
    model.add(layers.Conv2D(128, kernel_size=5, strides=2, padding='same'))
    model.add(layers.LeakyReLU(alpha=0.3))
    model.add(layers.Dropout(0.3))

    # Flatten and map to scalar output
    model.add(layers.Flatten())
    model.add(layers.Dense(1))

    return model

In [ ]:
# Initialize models
generator = build_generator()
discriminator = build_discriminator()


In [ ]:
# Define the loss function
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)


In [ ]:
# Generator loss: we want the discriminator to identify fake images as real
def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

# Discriminator loss: correctly identify real as real (1) and fake as fake (0)
def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss


In [ ]:
# Define optimizers with learning rate 10^-4
generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)


In [ ]:
# Create a fixed seed for image generation to see progress
seed = tf.random.normal([images_generated, z_dim])

In [ ]:
# Function to generate and display images
def generate_and_display_images(model, test_input, epoch):
    # Generate images from the seed
    predictions = model(test_input, training=False)

    # Create a 3x3 grid of generated images
    fig = plt.figure(figsize=(3, 3))

    for i in range(predictions.shape[0]):
        plt.subplot(3, 3, i+1)
        # Scale back to [0,255]
        plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5, cmap='gray')
        plt.axis('off')

    plt.suptitle(f'Epoch: {epoch}')
    plt.tight_layout()
    plt.show()

In [ ]:
# Training step
@tf.function
def train_step(images):
    noise = tf.random.normal([images.shape[0], z_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        # Generate fake images
        generated_images = generator(noise, training=True)

        # Get discriminator outputs for real and fake images
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        # Calculate losses
        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    # Calculate gradients
    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    # Apply gradients
    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss

In [ ]:
# Training function
def train(dataset, epochs):
    generator_losses = []
    discriminator_losses = []

    for epoch in range(epochs):
        print(f'Starting epoch {epoch+1}/{epochs}')

        epoch_gen_loss = 0
        epoch_disc_loss = 0
        steps = 0

        for image_batch in dataset:
            g_loss, d_loss = train_step(image_batch)
            epoch_gen_loss += g_loss
            epoch_disc_loss += d_loss
            steps += 1

        # Calculate average loss for this epoch
        epoch_gen_loss /= steps
        epoch_disc_loss /= steps

        generator_losses.append(float(epoch_gen_loss))
        discriminator_losses.append(float(epoch_disc_loss))

        # Print status
        print(f'Epoch {epoch+1}, Generator Loss: {epoch_gen_loss}, Discriminator Loss: {epoch_disc_loss}')

        # Generate and display intermediate results at epochs 10, 30, and 50
        if (epoch + 1) == 10 or (epoch + 1) == 30 or (epoch + 1) == 50:
            generate_and_display_images(generator, seed, epoch + 1)

    return generator_losses, discriminator_losses

In [ ]:
# Train the model
generator_losses, discriminator_losses = train(train_dataset, EPOCHS)

In [ ]:
# Plot the loss curves
plt.figure(figsize=(10, 6))
plt.plot(range(EPOCHS), generator_losses, label='Generator Loss')
plt.plot(range(EPOCHS), discriminator_losses, label='Discriminator Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.title('Generator and Discriminator Loss')
plt.show()


In [ ]:
# Generate a final set of images
generate_and_display_images(generator, seed, EPOCHS)